[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/da-ds-notebook-solutions/blob/main/LATAM%20V7%20DADS/9-18%20DS/SP17%20Computer%20Vision/P%20Spanish%20version/S17%20ESP%20SOL3%20Vision%20artificial%20-%20Colab%20GPU.ipynb)

> **Nota:** Este notebook requiere GPU para entrenar el modelo. Haz clic en el botón de arriba para abrirlo en Google Colab con acceso gratuito a GPU.

# Visión Artificial para Determinar la Edad

## Contexto del Proyecto

La cadena de supermercados **Good Seed** está interesada en cumplir con las leyes sobre el alcohol, asegurándose de no vender alcohol a personas menores de edad. Sus tiendas están equipadas con cámaras en el área de pago, las cuales se activan cuando una persona está comprando alcohol.

## Objetivo

Construir un modelo de **visión artificial** que determine la edad de una persona a partir de una fotografía. El modelo debe alcanzar un **MAE (Error Absoluto Medio) menor a 8 años** en el conjunto de validación.

## Dataset

- **7,591 imágenes** de rostros
- Etiquetas con edad real (1-100 años)
- Ubicación: `faces_dataset/`

---

<div style="background: linear-gradient(90deg, #ff6b6b 0%, #feca57 100%); padding: 20px; border-radius: 10px; margin: 20px 0;">
    <h2 style="color: white; margin: 0;">🎓 SECCIÓN EDUCATIVA</h2>
    <p style="color: #f0f0f0; margin: 5px 0 0 0;">Conceptos fundamentales antes de comenzar</p>
</div>

## ¿Qué es una GPU y Por Qué la Necesitamos?

### CPU vs GPU

| Característica | CPU | GPU |
|----------------|-----|-----|
| **Núcleos** | 4-16 núcleos potentes | Miles de núcleos pequeños |
| **Diseño** | Tareas secuenciales complejas | Tareas paralelas simples |
| **Analogía** | Un chef experto cocinando | 1000 ayudantes haciendo tareas simples |

### ¿Por qué las CNNs necesitan GPU?

Las redes neuronales convolucionales (CNNs) realizan **millones de operaciones matriciales** por imagen:

- Una imagen de 224x224 píxeles = **150,528 valores**
- ResNet50 tiene ~**25 millones de parámetros**
- Por cada imagen, se calculan millones de multiplicaciones

### Comparación de Tiempos

| Hardware | Tiempo por Época (7,591 imágenes) |
|----------|-----------------------------------|
| CPU | ~30-60 minutos |
| GPU (T4) | ~38 segundos |

**Conclusión:** Sin GPU, entrenar 20 épocas tomaría **10-20 horas**. Con GPU, toma **~13 minutos**.

## ¿Qué es ResNet50?

**ResNet** (Residual Network) es una arquitectura de red neuronal que ganó la competencia **ImageNet 2015**.

### El Problema que Resuelve

Antes de ResNet, las redes muy profundas (muchas capas) sufrían de **vanishing gradient**: los gradientes se hacían tan pequeños que las primeras capas dejaban de aprender.

### La Solución: Skip Connections

ResNet introduce **conexiones residuales** (skip connections) que permiten que la información fluya directamente a través de las capas:

```
Entrada ─────────────────────────────┐
   │                                 │
   ▼                                 │
┌──────────────┐                     │
│   Conv 3x3   │                     │
│   BatchNorm  │                     │
│   ReLU       │                     │
└──────────────┘                     │
   │                                 │
   ▼                                 │
┌──────────────┐                     │
│   Conv 3x3   │                     │
│   BatchNorm  │                     │
└──────────────┘                     │
   │                                 │
   ▼                                 │
   ⊕ ◄───────────────────────────────┘  (Skip Connection)
   │
   ▼
  ReLU
   │
   ▼
 Salida
```

### Transfer Learning

Usamos ResNet50 **pre-entrenado en ImageNet** (1.2 millones de imágenes, 1000 clases). Esto significa:

1. La red ya sabe detectar bordes, texturas, formas
2. Solo necesitamos "afinar" las últimas capas para nuestra tarea específica (predecir edad)
3. Entrenamiento mucho más rápido y con menos datos

## Parámetros de Entrenamiento Explicados

| Parámetro | Valor | ¿Qué significa? |
|-----------|-------|------------------|
| `batch_size` | 32 | Procesamos 32 imágenes antes de actualizar los pesos. Mayor batch = más rápido pero requiere más memoria GPU |
| `epochs` | 20 | El modelo ve TODO el dataset 20 veces. Más épocas = más aprendizaje, pero riesgo de overfitting |
| `steps_per_epoch` | 178 | = 5,694 imágenes / 32 batch_size. Cuántos batches por época |
| `validation_split` | 0.25 | 75% entrenamiento, 25% validación |
| `target_size` | (224, 224) | Tamaño estándar para ResNet. Todas las imágenes se redimensionan |
| `rescale` | 1./255 | Normaliza píxeles de 0-255 a 0-1 (las redes funcionan mejor con valores pequeños) |

## Optimizador Adam

**Adam** (Adaptive Moment Estimation) es el optimizador más popular en deep learning.

### ¿Qué hace un optimizador?

Ajusta los **pesos** de la red para minimizar el error. Imagina que estás en una montaña con niebla y quieres llegar al punto más bajo:

- **SGD básico**: Das pasos del mismo tamaño siempre
- **Adam**: Ajusta el tamaño del paso según el terreno (más rápido en pendientes suaves, más cuidadoso en zonas empinadas)

### Learning Rate

```python
Adam(learning_rate=0.0001)
```

- **Learning rate alto (0.01)**: Pasos grandes, aprende rápido pero puede "saltarse" el óptimo
- **Learning rate bajo (0.0001)**: Pasos pequeños, más preciso pero más lento

Para **transfer learning**, usamos LR bajo porque los pesos ya están cerca del óptimo.

## Función de Pérdida y Métrica

### MSE (Mean Squared Error) - Función de Pérdida

```python
loss = 'mean_squared_error'
```

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Penaliza errores grandes más que errores pequeños (por el cuadrado).

### MAE (Mean Absolute Error) - Métrica

```python
metrics = ['mae']
```

$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Más interpretable: "El modelo se equivoca en promedio X años".

**Objetivo del proyecto: MAE < 8 años**

---

<div style="background: linear-gradient(90deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; margin: 20px 0;">
    <h2 style="color: white; margin: 0;">🖥️ CÓDIGO LOCAL - Jupyter Notebook</h2>
    <p style="color: #f0f0f0; margin: 5px 0 0 0;">Esta sección se puede ejecutar en tu máquina local (no requiere GPU)</p>
</div>

## Inicialización

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Carga de Datos

El dataset contiene:
- Carpeta `final_files/` con 7,591 imágenes JPG
- Archivo `labels.csv` con las columnas `file_name` y `real_age`

In [ ]:
# Ajustar path según donde esté tu dataset
# En Colab: path = '/content/faces_dataset/'
# En local: path = 'datasets/faces_dataset/'  o la ruta donde lo tengas

path = 'datasets/faces_dataset/'

labels = pd.read_csv(path + 'labels.csv')
print(f"Total de imágenes: {len(labels)}")
labels.head()

## Análisis Exploratorio de Datos (EDA)

In [ ]:
# Información general
labels.info()

In [ ]:
# Estadísticas descriptivas
labels.describe()

In [ ]:
# Distribución de edades
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(labels['real_age'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Edades')
axes[0].axvline(labels['real_age'].mean(), color='red', linestyle='--', label=f'Media: {labels["real_age"].mean():.1f}')
axes[0].axvline(labels['real_age'].median(), color='green', linestyle='--', label=f'Mediana: {labels["real_age"].median():.1f}')
axes[0].legend()

# Box plot
axes[1].boxplot(labels['real_age'], vert=True)
axes[1].set_ylabel('Edad')
axes[1].set_title('Box Plot de Edades')

plt.tight_layout()
plt.show()

In [ ]:
# Análisis de menores de edad (importante para el objetivo del proyecto)
menores = labels[labels['real_age'] < 18]
adultos = labels[labels['real_age'] >= 18]

print(f"Menores de 18 años: {len(menores)} ({len(menores)/len(labels)*100:.1f}%)")
print(f"Adultos (18+): {len(adultos)} ({len(adultos)/len(labels)*100:.1f}%)")

# Desglose de menores por edad
print("\nDesglose de menores (11-17 años):")
print(labels[(labels['real_age'] >= 11) & (labels['real_age'] <= 17)].groupby('real_age').size())

In [ ]:
# Vista previa de imágenes
train_datagen = ImageDataGenerator(rescale=1./255)

sample_gen = train_datagen.flow_from_dataframe(
    dataframe=labels,
    directory=path + 'final_files/',
    x_col='file_name',
    y_col='real_age',
    target_size=(224, 224),
    batch_size=16,
    class_mode='raw',
    seed=12345
)

features, target = next(sample_gen)

fig = plt.figure(figsize=(12, 8))
for i in range(16):
    ax = fig.add_subplot(4, 4, i+1)
    ax.imshow(features[i])
    ax.set_title(f'Edad: {int(target[i])}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Conclusiones del EDA

- **Rango de edades:** 1 a 100 años
- **Media:** ~31 años | **Mediana:** ~29 años
- **Distribución:** Sesgada hacia adultos jóvenes (20-40 años)
- **Menores de edad:** Solo ~25% del dataset tiene menos de 20 años
- **Desafío:** Poca representación de menores de edad, que son precisamente el grupo crítico para el objetivo del proyecto

---

<div style="background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); padding: 20px; border-radius: 10px; margin: 20px 0;">
    <h2 style="color: white; margin: 0;">⚡ CÓDIGO GPU - Google Colab</h2>
    <p style="color: #f0f0f0; margin: 5px 0 0 0;">A partir de aquí, ejecutar en Google Colab con GPU habilitada</p>
</div>

### Instrucciones para Colab:

1. **Abre este notebook en Colab** (botón al inicio)
2. **Activa la GPU:** `Runtime → Change runtime type → T4 GPU`
3. **Ejecuta las celdas** en orden

In [ ]:
# ============================================
# EJECUTAR EN GOOGLE COLAB
# ============================================

# 1. Verificar GPU
!nvidia-smi

In [ ]:
# 2. Descargar dataset desde GitHub Release
!wget -q --show-progress https://github.com/HesusG/da-ds-notebook-solutions/releases/download/asset/faces_dataset.zip
!unzip -q faces_dataset.zip
!rm faces_dataset.zip

# Verificar
!ls faces_dataset/

In [ ]:
# 3. Imports para entrenamiento
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adam

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Path para Colab
path = '/content/faces_dataset/'

## Funciones del Modelo

In [ ]:
def load_train(path):
    """
    Carga la parte de entrenamiento del conjunto de datos.
    
    Parámetros:
    - path: Ruta al directorio del dataset
    
    Retorna:
    - Generador de imágenes de entrenamiento
    """
    labels = pd.read_csv(path + 'labels.csv')
    
    train_datagen = ImageDataGenerator(
        rescale=1./255,           # Normalizar píxeles a 0-1
        validation_split=0.25     # 25% para validación
    )
    
    train_gen_flow = train_datagen.flow_from_dataframe(
        dataframe=labels,
        directory=path + 'final_files/',
        x_col='file_name',
        y_col='real_age',
        target_size=(224, 224),   # Tamaño estándar para ResNet
        batch_size=32,
        class_mode='raw',         # Regresión (no clasificación)
        subset='training',        # Subset de entrenamiento
        seed=12345
    )
    
    return train_gen_flow

In [ ]:
def load_test(path):
    """
    Carga la parte de validación del conjunto de datos.
    
    Parámetros:
    - path: Ruta al directorio del dataset
    
    Retorna:
    - Generador de imágenes de validación
    """
    labels = pd.read_csv(path + 'labels.csv')
    
    test_datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=0.25
    )
    
    test_gen_flow = test_datagen.flow_from_dataframe(
        dataframe=labels,
        directory=path + 'final_files/',
        x_col='file_name',
        y_col='real_age',
        target_size=(224, 224),
        batch_size=32,
        class_mode='raw',
        subset='validation',      # Subset de validación
        seed=12345
    )
    
    return test_gen_flow

In [ ]:
def create_model(input_shape=(224, 224, 3)):
    """
    Crea el modelo de CNN usando ResNet50 como backbone.
    
    Arquitectura:
    1. ResNet50 pre-entrenado (sin capa final)
    2. GlobalAveragePooling2D (reduce dimensiones)
    3. Dense(1) con ReLU (predicción de edad)
    
    Parámetros:
    - input_shape: Dimensiones de entrada (224, 224, 3)
    
    Retorna:
    - Modelo compilado
    """
    # Cargar ResNet50 pre-entrenado en ImageNet
    backbone = ResNet50(
        input_shape=input_shape,
        weights='imagenet',       # Pesos pre-entrenados
        include_top=False         # Sin capa de clasificación final
    )
    
    # Construir modelo
    model = Sequential([
        backbone,
        GlobalAveragePooling2D(),  # Reduce (7, 7, 2048) → (2048,)
        Dense(1, activation='relu') # Salida: edad predicha
    ])
    
    # Compilar
    model.compile(
        optimizer=Adam(learning_rate=0.0001),  # LR bajo para fine-tuning
        loss='mean_squared_error',              # Penaliza errores grandes
        metrics=['mae']                         # Error absoluto medio
    )
    
    return model

In [ ]:
def train_model(model, train_data, test_data, batch_size=None, epochs=20,
                steps_per_epoch=None, validation_steps=None):
    """
    Entrena el modelo.
    
    Parámetros:
    - model: Modelo a entrenar
    - train_data: Generador de entrenamiento
    - test_data: Generador de validación
    - epochs: Número de épocas (default: 20)
    
    Retorna:
    - Modelo entrenado
    """
    if steps_per_epoch is None:
        steps_per_epoch = len(train_data)
    if validation_steps is None:
        validation_steps = len(test_data)
    
    model.fit(
        train_data,
        validation_data=test_data,
        batch_size=batch_size,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        verbose=2  # Una línea por época
    )
    
    return model

## Entrenamiento del Modelo

In [ ]:
# Cargar datos
train_data = load_train(path)
test_data = load_test(path)

print(f"\nImágenes de entrenamiento: {train_data.samples}")
print(f"Imágenes de validación: {test_data.samples}")

In [ ]:
# Crear modelo
model = create_model(input_shape=(224, 224, 3))
model.summary()

In [ ]:
# Entrenar (esto toma ~13 minutos con GPU T4)
model = train_model(
    model,
    train_data,
    test_data,
    epochs=20
)

## Resultados del Entrenamiento

**Resultados típicos después de 20 épocas:**

```
Epoch 1/20  - loss: 245.10 - mae: 11.18 - val_loss: 894.18 - val_mae: 25.06
Epoch 5/20  - loss: 14.76  - mae: 2.99  - val_loss: 86.01  - val_mae: 6.94
Epoch 10/20 - loss: 7.00   - mae: 1.99  - val_loss: 65.94  - val_mae: 6.21
Epoch 15/20 - loss: 5.37   - mae: 1.74  - val_loss: 64.16  - val_mae: 6.07
Epoch 20/20 - loss: 5.54   - mae: 1.77  - val_loss: 68.34  - val_mae: 6.14
```

**MAE final en validación: ~6.14 años** ✅ (Objetivo: < 8 años)

---

## Conclusiones

### Resultados Obtenidos

- **MAE en validación:** ~6.14 años
- **Objetivo cumplido:** Sí (MAE < 8)
- **Tiempo de entrenamiento:** ~13 minutos (20 épocas con GPU T4)

### Interpretación

El modelo predice la edad de una persona con un **error promedio de ~6 años**. Esto significa:
- Si el modelo predice 20 años, la edad real probablemente está entre 14-26 años
- Para verificar mayoría de edad (18+), hay margen de error

### Limitaciones

1. **Desbalance de datos:** Solo 25% de las imágenes son de menores de 20 años
2. **Rango crítico:** El modelo puede confundir personas de 15-17 años con jóvenes de 18-21
3. **Overfitting leve:** MAE de entrenamiento (~1.8) vs validación (~6.1)

### Recomendaciones para Mejorar

1. **Más datos de menores:** Recolectar más imágenes de personas entre 14-20 años
2. **Data augmentation:** Rotaciones, zoom, cambios de brillo para aumentar variabilidad
3. **Fine-tuning:** Descongelar algunas capas de ResNet para mayor adaptación
4. **Callbacks:** Early stopping y reducción de learning rate para evitar overfitting

---

## Checklist del Proyecto

- [x] Cargar imágenes desde `/datasets/faces/final_files/`
- [x] Cargar etiquetas desde `labels.csv`
- [x] Realizar EDA con visualizaciones
- [x] Redimensionar imágenes a tamaño consistente (224x224)
- [x] Normalizar píxeles (rescale 1/255)
- [x] Usar ImageDataGenerator con flow_from_dataframe
- [x] Configurar validation_split correctamente
- [x] Usar transfer learning (ResNet50)
- [x] Capa de salida apropiada para regresión (Dense 1)
- [x] Entrenar por al menos 10 épocas
- [x] **MAE < 8 en validación** ✅